# 🧱 Lab: Agent Evaluation

**Module 9: LLM Evaluation** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** an evaluation framework that assesses agent task completion and tool use quality
2. **Apply** LLM-as-judge scoring to evaluate whether an agent chose the right tools and used them correctly
3. **Create** a custom rubric tailored to agent behavior — covering tool selection, parameter accuracy, and final answer quality
4. **Analyze** agent evaluation results to identify failure patterns across a test suite

## Concepts Covered

| Concept | From |
|---------|------|
| LLM-as-Judge | mini-llm-judge |
| Rubric Scoring | mini-auto-grader |
| Automated Evaluation Pipeline | lab-evaluation-pipeline |
| Agent Evaluation | This lab |

## Prerequisites

- **mini-llm-judge**: Using an LLM to evaluate output quality with pointwise scoring
- **mini-auto-grader**: Designing custom rubrics with weighted criteria and score-level definitions
- **lab-evaluation-pipeline**: Building multi-layer evaluation pipelines with pass/fail reporting
- OpenAI API key configured in `.env`

## Why Agent Evaluation Is Different

In `lab-evaluation-pipeline`, we evaluated **text outputs** — does the generated answer match the reference? For agents, evaluation is harder because we need to assess the **process**, not just the result.

| Text Evaluation | Agent Evaluation |
|-----------------|------------------|
| Did the answer match the reference? | Did the agent pick the right tool(s)? |
| Was it accurate and complete? | Were the tool parameters correct? |
| One output to score | Multiple steps to trace |

An agent might give the correct final answer but use the wrong tool to get there (brittle). Or it might use the right tool with wrong parameters (lucky result). A proper agent evaluation checks **three dimensions**:

1. **Tool Selection** — Did the agent choose the correct tool(s)?
2. **Parameter Accuracy** — Did it pass the right arguments to each tool?
3. **Final Answer Quality** — Is the end result correct and useful?

## Setup

In [4]:
import os
import re
import json
from dotenv import load_dotenv
import openai
from IPython.display import display, Markdown

load_dotenv()

client = openai.OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

print("Setup complete.")

Setup complete.


## Simulated Agent Traces

Instead of running a live agent, we'll work with **pre-recorded agent traces** — the sequence of tool calls and final answers an agent produced for each task. This is exactly how agent evaluation works in practice: you capture traces in production or testing, then evaluate them offline.

Our simulated agent has access to three tools:

| Tool | Purpose |
|------|---------|
| `search_products` | Search the product catalog by query |
| `get_order_status` | Look up an order by order ID |
| `calculate_shipping` | Estimate shipping cost given weight and destination |

In [6]:
available_tools = [
    {"name": "search_products", "params": ["query"]},
    {"name": "get_order_status", "params": ["order_id"]},
    {"name": "calculate_shipping", "params": ["weight_kg", "destination"]}
]

agent_traces = [
    {
        "id": "T1",
        "task": "What's the status of order #12345?",
        "expected_tools": ["get_order_status"],
        "expected_params": {"get_order_status": {"order_id": "12345"}},
        "reference_answer": "Order #12345 is currently in transit and expected to arrive by Friday.",
        "agent_tools_called": [
            {"tool": "get_order_status", "params": {"order_id": "12345"}}
        ],
        "agent_answer": "Your order #12345 is in transit and should arrive by Friday."
    },
    {
        "id": "T2",
        "task": "How much would it cost to ship a 3kg package to London?",
        "expected_tools": ["calculate_shipping"],
        "expected_params": {"calculate_shipping": {"weight_kg": 3, "destination": "London"}},
        "reference_answer": "Shipping a 3kg package to London costs $18.50 with standard delivery.",
        "agent_tools_called": [
            {"tool": "search_products", "params": {"query": "shipping London"}}
        ],
        "agent_answer": "I found some products related to shipping. Let me know if you need help."
    },
    {
        "id": "T3",
        "task": "Find me wireless headphones and check how much shipping would be for a 0.5kg package to Berlin.",
        "expected_tools": ["search_products", "calculate_shipping"],
        "expected_params": {
            "search_products": {"query": "wireless headphones"},
            "calculate_shipping": {"weight_kg": 0.5, "destination": "Berlin"}
        },
        "reference_answer": "I found 3 wireless headphones options. Shipping 0.5kg to Berlin costs $12.00.",
        "agent_tools_called": [
            {"tool": "search_products", "params": {"query": "wireless headphones"}},
            {"tool": "calculate_shipping", "params": {"weight_kg": 0.5, "destination": "Berlin"}}
        ],
        "agent_answer": "Here are some wireless headphones options. Shipping a 0.5kg package to Berlin would cost $12.00."
    },
    {
        "id": "T4",
        "task": "What's the status of order #99887?",
        "expected_tools": ["get_order_status"],
        "expected_params": {"get_order_status": {"order_id": "99887"}},
        "reference_answer": "Order #99887 was delivered on Tuesday.",
        "agent_tools_called": [
            {"tool": "get_order_status", "params": {"order_id": "9988"}}
        ],
        "agent_answer": "I couldn't find that order. Please double-check the order number."
    },
    {
        "id": "T5",
        "task": "Search for running shoes.",
        "expected_tools": ["search_products"],
        "expected_params": {"search_products": {"query": "running shoes"}},
        "reference_answer": "Several running shoes: TrailRunner Pro ($89), SpeedLite 3 ($120), ComfortRun ($65).",
        "agent_tools_called": [
            {"tool": "search_products", "params": {"query": "running shoes"}}
        ],
        "agent_answer": "I found several running shoes: TrailRunner Pro ($89), SpeedLite 3 ($120), and ComfortRun ($65). Would you like details on any of these?"
    }
]

print(f"Agent traces: {len(agent_traces)} test cases")
print(f"Available tools: {[t['name'] for t in available_tools]}")

Agent traces: 5 test cases
Available tools: ['search_products', 'get_order_status', 'calculate_shipping']


Let's review what the traces look like. Each has a mix of quality — some correct, some with the wrong tool, some with wrong parameters:

| Trace | Expected Behavior | Actual Agent Behavior | Issue |
|-------|-------------------|-----------------------|-------|
| T1 | `get_order_status(12345)` | ✅ Correct tool + params | None |
| T2 | `calculate_shipping(3, London)` | ❌ Used `search_products` instead | Wrong tool |
| T3 | `search_products` + `calculate_shipping` | ✅ Both correct | None |
| T4 | `get_order_status(99887)` | ⚠️ Right tool, wrong param (`9988`) | Truncated order ID |
| T5 | `search_products(running shoes)` | ✅ Correct tool + params | None |

## Deterministic Checks: Tool Selection & Parameters

Before spending API calls on LLM-as-judge, we can run **deterministic checks** on tool selection and parameter accuracy. These are fast, free, and catch obvious failures.

In [7]:
def check_tool_selection(trace):
    """Check if the agent called the expected tools."""
    expected = set(trace["expected_tools"])
    actual = set(t["tool"] for t in trace["agent_tools_called"])
    
    correct = expected == actual
    missing = expected - actual
    extra = actual - expected
    
    return {
        "correct": correct,
        "expected": expected,
        "actual": actual,
        "missing": missing,
        "extra": extra
    }


def check_parameters(trace):
    """Check if tool parameters match expected values."""
    issues = []
    for call in trace["agent_tools_called"]:
        tool_name = call["tool"]
        if tool_name in trace["expected_params"]:
            expected_p = trace["expected_params"][tool_name]
            actual_p = call["params"]
            for key, exp_val in expected_p.items():
                act_val = actual_p.get(key)
                if str(act_val) != str(exp_val):
                    issues.append(f"{tool_name}.{key}: expected '{exp_val}', got '{act_val}'")
    return {"correct": len(issues) == 0, "issues": issues}

In [8]:
# Run deterministic checks on all traces
for trace in agent_traces:
    trace["tool_check"] = check_tool_selection(trace)
    trace["param_check"] = check_parameters(trace)

lines = [
    "### Deterministic Checks\n",
    "| ID | Tool Selection | Parameters | Issues |",
    "|----|---------------|------------|--------|"
]
for t in agent_traces:
    tool_ok = "✅" if t["tool_check"]["correct"] else "❌"
    param_ok = "✅" if t["param_check"]["correct"] else "❌"
    issues = "; ".join(
        [f"Missing: {t['tool_check']['missing']}" for _ in [1] if t['tool_check']['missing']] +
        [f"Extra: {t['tool_check']['extra']}" for _ in [1] if t['tool_check']['extra']] +
        t["param_check"]["issues"]
    ) or "None"
    lines.append(f"| {t['id']} | {tool_ok} | {param_ok} | {issues} |")

md("\n".join(lines))

### Deterministic Checks

| ID | Tool Selection | Parameters | Issues |
|----|---------------|------------|--------|
| T1 | ✅ | ✅ | None |
| T2 | ❌ | ✅ | Missing: {'calculate_shipping'}; Extra: {'search_products'} |
| T3 | ✅ | ✅ | None |
| T4 | ✅ | ❌ | get_order_status.order_id: expected '99887', got '9988' |
| T5 | ✅ | ✅ | None |

Deterministic checks instantly flag T2 (wrong tool) and T4 (wrong parameter). But they can't tell us about the **quality of the final answer** — that's where the LLM judge comes in.

## Agent Evaluation Rubric

Recall from `mini-auto-grader`: a rubric defines criteria, score levels, and weights. For agent evaluation, our criteria map to the three dimensions we identified earlier: tool selection, parameter accuracy, and final answer quality.

We give **tool selection** the highest weight because using the wrong tool means the agent fundamentally misunderstood the task.

In [9]:
agent_rubric = {
    "name": "Agent Task Completion",
    "criteria": [
        {
            "name": "Tool Selection",
            "description": "Did the agent choose the correct tool(s) for the task?",
            "weight": 0.4,
            "levels": {
                1: "Used completely wrong tool(s) — no overlap with expected",
                2: "Partially correct — got one tool right but missed others or added unnecessary ones",
                3: "Correct tools but also called an unnecessary extra tool",
                4: "All correct tools selected with one minor issue",
                5: "Perfect tool selection — exactly the right tools, nothing extra"
            }
        },
        {
            "name": "Parameter Accuracy",
            "description": "Were the correct parameters passed to each tool?",
            "weight": 0.3,
            "levels": {
                1: "Parameters completely wrong or missing",
                2: "Major parameter errors that would cause tool failure",
                3: "Partially correct parameters — some right, some wrong",
                4: "Nearly correct with only minor issues (e.g., formatting)",
                5: "All parameters exactly correct"
            }
        },
        {
            "name": "Answer Quality",
            "description": "Is the final answer correct, helpful, and responsive to the user's task?",
            "weight": 0.3,
            "levels": {
                1: "Answer is wrong or completely unrelated to the task",
                2: "Answer is vague or only partially addresses the task",
                3: "Answer addresses the task but missing key details",
                4: "Good answer with minor omissions",
                5: "Excellent — fully answers the task with all relevant details"
            }
        }
    ]
}

print(f"Rubric: {agent_rubric['name']}")
print(f"Criteria: {[c['name'] for c in agent_rubric['criteria']]}")
print(f"Weights sum: {sum(c['weight'] for c in agent_rubric['criteria'])}")

Rubric: Agent Task Completion
Criteria: ['Tool Selection', 'Parameter Accuracy', 'Answer Quality']
Weights sum: 1.0


## LLM-as-Judge for Agent Traces

The judge prompt needs to include the full agent trace — what tools were available, what the agent called, and what it answered — so the judge can assess the entire process.

In [10]:
def build_agent_judge_prompt(rubric, trace, tools):
    """Build a judge prompt for evaluating an agent trace."""
    # Format rubric criteria
    criteria_text = ""
    for c in rubric["criteria"]:
        levels_text = "\n".join(f"      {k}: {v}" for k, v in c["levels"].items())
        criteria_text += (
            f"\n  - **{c['name']}** (weight: {c['weight']})\n"
            f"    {c['description']}\n"
            f"    Score levels:\n{levels_text}\n"
        )

    format_lines = "\n".join(
        f"{c['name']}: <score>/5 - <justification>" for c in rubric["criteria"]
    )

    tools_desc = "\n".join(f"  - {t['name']}({', '.join(t['params'])})" for t in tools)
    
    agent_calls = "\n".join(
        f"  - {c['tool']}({json.dumps(c['params'])})" for c in trace["agent_tools_called"]
    )

    return f"""You are an expert evaluator assessing an AI agent's performance.

**User Task:** {trace['task']}

**Available Tools:**
{tools_desc}

**Expected Tool Calls:** {trace['expected_tools']}
**Expected Parameters:** {json.dumps(trace['expected_params'])}

**Agent's Actual Tool Calls:**
{agent_calls}

**Agent's Final Answer:** {trace['agent_answer']}
**Reference Answer:** {trace.get('reference_answer', '(not provided)')}

**Rubric: {rubric['name']}**
{criteria_text}
Score the agent on EACH criterion. Use the score levels above.

Format your answer EXACTLY like this:
{format_lines}
"""

In [11]:
def judge_agent_trace(rubric, trace, tools, model="gpt-4o-mini"):
    """Grade an agent trace using the rubric-based LLM judge."""
    prompt = build_agent_judge_prompt(rubric, trace, tools)
    
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    judgment = result.choices[0].message.content
    
    scores = {}
    for c in rubric["criteria"]:
        match = re.search(rf"{c['name']}:\s*(\d)/5\s*-\s*(.+)", judgment)
        if match:
            scores[c["name"]] = {
                "score": int(match.group(1)),
                "justification": match.group(2).strip(),
                "weight": c["weight"]
            }
    
    weighted_total = sum(s["score"] * s["weight"] for s in scores.values())
    return {"scores": scores, "weighted_total": weighted_total, "raw": judgment}

print("Agent judge function ready.")

Agent judge function ready.


## Running the Agent Evaluation Pipeline

Now we combine both layers — deterministic checks and rubric-based LLM judging — into one pipeline run across all traces.

In [12]:
PASS_THRESHOLD = 3.5

for trace in agent_traces:
    trace["rubric_result"] = judge_agent_trace(agent_rubric, trace, available_tools)
    trace["passed"] = trace["rubric_result"]["weighted_total"] >= PASS_THRESHOLD

print("All traces evaluated.")


All traces evaluated.


In [13]:
lines = [
    "## 📊 Agent Evaluation Report\n",
    "| ID | Tool Check | Param Check | Tool Selection | Param Accuracy | Answer Quality | Weighted | Pass? |",
    "|----|-----------|-------------|---------------|---------------|---------------|----------|-------|"
]

pass_count = 0
for t in agent_traces:
    tc = "✅" if t["tool_check"]["correct"] else "❌"
    pc = "✅" if t["param_check"]["correct"] else "❌"
    r = t["rubric_result"]["scores"]
    ts = r.get("Tool Selection", {}).get("score", "?")
    pa = r.get("Parameter Accuracy", {}).get("score", "?")
    aq = r.get("Answer Quality", {}).get("score", "?")
    wt = t["rubric_result"]["weighted_total"]
    passed = t["passed"]
    if passed:
        pass_count += 1
    lines.append(
        f"| {t['id']} | {tc} | {pc} | {ts}/5 | {pa}/5 | {aq}/5 | {wt:.2f}/5 | {'✅' if passed else '❌'} |"
    )

lines.append(f"\n**Pass rate: {pass_count}/{len(agent_traces)} ({pass_count/len(agent_traces)*100:.0f}%)** (threshold: {PASS_THRESHOLD}/5)")

md("\n".join(lines))

## 📊 Agent Evaluation Report

| ID | Tool Check | Param Check | Tool Selection | Param Accuracy | Answer Quality | Weighted | Pass? |
|----|-----------|-------------|---------------|---------------|---------------|----------|-------|
| T1 | ✅ | ✅ | 5/5 | 5/5 | 5/5 | 5.00/5 | ✅ |
| T2 | ❌ | ✅ | 1/5 | 1/5 | 1/5 | 1.00/5 | ❌ |
| T3 | ✅ | ✅ | 5/5 | 5/5 | 4/5 | 4.70/5 | ✅ |
| T4 | ✅ | ❌ | 4/5 | 2/5 | 2/5 | 2.80/5 | ❌ |
| T5 | ✅ | ✅ | 5/5 | 5/5 | 5/5 | 5.00/5 | ✅ |

**Pass rate: 3/5 (60%)** (threshold: 3.5/5)

## Drill-Down: Inspecting Agent Failures

Just like in `lab-evaluation-pipeline`, let's surface the failures with full details so we can diagnose *what went wrong* in the agent's reasoning.

In [14]:
failures = [t for t in agent_traces if not t["passed"]]

if not failures:
    md("**All traces passed!** No failures to inspect.")
else:
    lines = [f"### ❌ {len(failures)} Failing Trace(s)\n"]
    for t in failures:
        lines.append("---\n")
        lines.append(f"**{t['id']}: {t['task']}**\n")
        lines.append(f"- **Expected tools:** {t['expected_tools']}")
        actual_tools = [c['tool'] for c in t['agent_tools_called']]
        lines.append(f"- **Actual tools:** {actual_tools}")
        lines.append(f"- **Agent answer:** {t['agent_answer']}")
        lines.append(f"- **Reference answer:** {t.get('reference_answer', '(not provided)')}\n")
        lines.append("| Criterion | Score | Why |")
        lines.append("|-----------|-------|-----|")
        for name, data in t["rubric_result"]["scores"].items():
            lines.append(f"| {name} | {data['score']}/5 | {data['justification']} |")
        lines.append(f"\n**Weighted Total: {t['rubric_result']['weighted_total']:.2f}/5**\n")
    
    md("\n".join(lines))

### ❌ 2 Failing Trace(s)

---

**T2: How much would it cost to ship a 3kg package to London?**

- **Expected tools:** ['calculate_shipping']
- **Actual tools:** ['search_products']
- **Agent answer:** I found some products related to shipping. Let me know if you need help.
- **Reference answer:** Shipping a 3kg package to London costs $18.50 with standard delivery.

| Criterion | Score | Why |
|-----------|-------|-----|
| Tool Selection | 1/5 | The agent used a completely wrong tool (search_products) that does not address the user's request for shipping cost, which should have been handled by the calculate_shipping tool. |
| Parameter Accuracy | 1/5 | Since the agent did not call the correct tool, there were no parameters passed, resulting in a complete failure to provide any relevant information. |
| Answer Quality | 1/5 | The final answer is unrelated to the user's task of determining the shipping cost, providing no useful information or assistance regarding the request. |

**Weighted Total: 1.00/5**

---

**T4: What's the status of order #99887?**

- **Expected tools:** ['get_order_status']
- **Actual tools:** ['get_order_status']
- **Agent answer:** I couldn't find that order. Please double-check the order number.
- **Reference answer:** Order #99887 was delivered on Tuesday.

| Criterion | Score | Why |
|-----------|-------|-----|
| Tool Selection | 4/5 | The agent selected the correct tool, `get_order_status`, but made an error in the order ID by using "9988" instead of "99887". This is a minor issue since the correct tool was chosen. |
| Parameter Accuracy | 2/5 | The parameters passed to the tool were incorrect. The order ID was significantly wrong, which would lead to a failure in retrieving the correct order status. |
| Answer Quality | 2/5 | The final answer was not correct as it failed to provide the status of order #99887. Instead, it suggested that the order could not be found, which does not help the user. The answer was vague and did not address the user's request adequately. |

**Weighted Total: 2.80/5**


## Failure Pattern Analysis

With a larger test suite, you'd want to aggregate results to spot **systemic issues**. Even with five traces, we can compute per-criterion averages to see where the agent struggles most.

In [15]:
# Aggregate scores by criterion
criterion_totals = {}
for t in agent_traces:
    for name, data in t["rubric_result"]["scores"].items():
        if name not in criterion_totals:
            criterion_totals[name] = []
        criterion_totals[name].append(data["score"])

lines = [
    "### Per-Criterion Averages\n",
    "| Criterion | Avg Score | Min | Max | Interpretation |",
    "|-----------|-----------|-----|-----|----------------|"
]

for name, scores in criterion_totals.items():
    avg = sum(scores) / len(scores)
    mn = min(scores)
    mx = max(scores)
    if avg >= 4.0:
        interp = "Strong ✅"
    elif avg >= 3.0:
        interp = "Needs attention ⚠️"
    else:
        interp = "Weak — fix this ❌"
    lines.append(f"| {name} | {avg:.1f}/5 | {mn} | {mx} | {interp} |")

# Overall
all_weighted = [t["rubric_result"]["weighted_total"] for t in agent_traces]
lines.append(f"\n**Overall weighted average: {sum(all_weighted)/len(all_weighted):.2f}/5**")

md("\n".join(lines))

### Per-Criterion Averages

| Criterion | Avg Score | Min | Max | Interpretation |
|-----------|-----------|-----|-----|----------------|
| Tool Selection | 4.0/5 | 1 | 5 | Strong ✅ |
| Parameter Accuracy | 3.6/5 | 1 | 5 | Needs attention ⚠️ |
| Answer Quality | 3.4/5 | 1 | 5 | Needs attention ⚠️ |

**Overall weighted average: 3.70/5**

## The Reusable Agent Evaluation Function

Let's package everything into a single function — just like we did with `evaluate_pipeline` in the previous lab. This pattern lets you run agent evaluation after every prompt change, model swap, or tool update.

In [16]:
def evaluate_agent(traces, rubric, tools, pass_threshold=3.5, model="gpt-4o-mini"):
    """Run the full agent evaluation pipeline on a set of traces."""
    results = []
    
    for trace in traces:
        # Deterministic checks
        tool_check = check_tool_selection(trace)
        param_check = check_parameters(trace)
        
        # LLM rubric grading
        rubric_result = judge_agent_trace(rubric, trace, tools, model=model)
        
        passed = rubric_result["weighted_total"] >= pass_threshold
        
        results.append({
            "id": trace["id"],
            "task": trace["task"],
            "tool_check": tool_check,
            "param_check": param_check,
            "rubric_result": rubric_result,
            "passed": passed
        })
    
    pass_rate = sum(1 for r in results if r["passed"]) / len(results)
    return {"results": results, "pass_rate": pass_rate}

print("evaluate_agent() ready.")

evaluate_agent() ready.


In [17]:
# Quick verification with first 2 traces
quick_test = evaluate_agent(agent_traces[:2], agent_rubric, available_tools)

lines = ["### Quick Test (2 traces)\n"]
for r in quick_test["results"]:
    lines.append(
        f"- **{r['id']}**: Tools={'✅' if r['tool_check']['correct'] else '❌'}, "
        f"Params={'✅' if r['param_check']['correct'] else '❌'}, "
        f"Rubric={r['rubric_result']['weighted_total']:.2f}/5, "
        f"{'✅ Pass' if r['passed'] else '❌ Fail'}"
    )
lines.append(f"\n**Pass rate: {quick_test['pass_rate']:.0%}**")

md("\n".join(lines))

### Quick Test (2 traces)

- **T1**: Tools=✅, Params=✅, Rubric=5.00/5, ✅ Pass
- **T2**: Tools=❌, Params=✅, Rubric=1.00/5, ❌ Fail

**Pass rate: 50%**

## Agent Eval vs. Text Eval: Key Differences

| Dimension | Text Evaluation (lab-evaluation-pipeline) | Agent Evaluation (this lab) |
|-----------|-------------------------------------------|-----------------------------|
| **What's evaluated** | A single generated text | A trace of tool calls + final answer |
| **Deterministic layer** | BLEU, ROUGE (word overlap) | Tool selection + parameter matching |
| **LLM judge focus** | Accuracy, completeness of text | Process quality — did the agent reason correctly? |
| **Rubric criteria** | Factual accuracy, actionability, conciseness | Tool selection, parameter accuracy, answer quality |
| **Failure modes** | Wrong facts, missing info | Wrong tool, wrong params, cascading errors |

## 🎯 Summary

### What You Built

You built an agent evaluation pipeline that assesses AI agent performance across two layers — deterministic checks for tool selection and parameter accuracy, and rubric-based LLM judging for holistic quality — then combines them into a pass/fail report with failure drill-down and pattern analysis.

### Key Takeaways

1. **Agent evaluation assesses process, not just output** — unlike text evaluation, you need to check which tools were selected, what parameters were passed, and whether the final answer is correct
2. **Deterministic checks are a free first layer** — comparing expected vs. actual tool calls and parameters catches obvious failures without any API calls
3. **Agent rubrics encode what matters** — weighting tool selection highest reflects that picking the wrong tool is a fundamental reasoning failure, worse than a minor parameter typo
4. **Failure pattern analysis reveals systemic issues** — aggregating per-criterion scores across traces shows whether the agent's weakness is tool selection, parameter handling, or answer generation, guiding targeted improvements